# 批量归一化 (Batch Normalization) — PyTorch 实现

> **PyTorch 版本** | [TensorFlow/Keras 版本](./Keras实现批量归一化.ipynb)

本 notebook 是 TensorFlow 版本的 PyTorch 等价实现，所有概念与数学原理保持一致，代码使用 PyTorch 框架。

## 核心思想

批量归一化（Ioffe & Szegedy, 2015）通过对每一层的输入进行归一化，解决深度网络训练中的**内部协变量偏移 (Internal Covariate Shift)** 问题。

## 数学原理

对于 mini-batch B = {x₁, x₂, ..., xₘ}：

1. **计算均值**: μ_B = (1/m) Σxᵢ
2. **计算方差**: σ²_B = (1/m) Σ(xᵢ - μ_B)²
3. **归一化**: x̂ᵢ = (xᵢ - μ_B) / √(σ²_B + ε)
4. **缩放平移**: yᵢ = γx̂ᵢ + β

其中 γ（scale/weight）和 β（shift/bias）是可学习参数，ε 是防止除零的小常数。

## 主要优势

| 优势 | 说明 |
|------|------|
| 加速训练 | 允许使用更大的学习率 |
| 稳定训练 | 减少梯度消失/爆炸 |
| 正则化效果 | 类似轻度 Dropout |
| 降低初始化敏感度 | 对权重初始化不那么敏感 |

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# 设置随机种子 / Set random seed
torch.manual_seed(42)
np.random.seed(42)

# 检查设备 / Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用设备: {device}")

## TF vs PyTorch 对照

在深入学习代码之前，先了解两个框架在批量归一化上的关键差异：

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 全连接层 BN | `keras.layers.BatchNormalization()` | `nn.BatchNorm1d(num_features)` |
| 卷积层 BN | `keras.layers.BatchNormalization()` | `nn.BatchNorm2d(num_features)` |
| 训练/推理切换 | `training=True/False` 参数 | `model.train()` / `model.eval()` |
| 滑动均值 | `moving_mean` | `running_mean` |
| 滑动方差 | `moving_variance` | `running_var` |
| 缩放参数 | `gamma` | `weight` |
| 偏移参数 | `beta` | `bias` |
| 无偏置线性层 | `Dense(..., use_bias=False)` | `nn.Linear(..., bias=False)` |

> **⚠️ momentum 参数差异（重要！）**
>
> 这是两个框架最容易踩坑的地方：
> - **Keras** 的 `momentum=0.99` 表示新统计量权重为 0.01，更新公式：`moving = moving * momentum + batch * (1 - momentum)`
> - **PyTorch** 的 `momentum=0.1` 表示新统计量权重为 0.1，更新公式：`running = running * (1 - momentum) + batch * momentum`
>
> 两者数学上等价，但数值**互为补数**！即 Keras 的 `0.99` 等价于 PyTorch 的 `0.01`，Keras 的 `0.9` 等价于 PyTorch 的 `0.1`。
>
> | Keras momentum | 等价 PyTorch momentum | 含义 |
> |---------------|----------------------|------|
> | 0.99 (默认) | 0.01 | 保留 99% 历史信息 |
> | 0.9 | 0.1 (默认) | 保留 90% 历史信息 |
> | 0.997 | 0.003 | 保留 99.7% 历史信息 |

## 1. 基本用法：激活函数后使用 BN

传统方式是在激活函数之后应用 BN。

In [ ]:
class ModelBNAfter(nn.Module):
    """
    带批量归一化的模型（标准方式：激活函数后使用 BN）
    Model with Batch Normalization after activation function.

    Architecture: Linear -> ELU -> BatchNorm1d -> ... -> Softmax
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        # 第一个 BN 层：归一化输入数据 / First BN layer: normalize input data
        self.bn_input = nn.BatchNorm1d(28 * 28)

        # 第一个隐藏层 / First hidden layer
        self.fc1 = nn.Linear(28 * 28, 300)
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='elu')
        self.elu1 = nn.ELU()
        self.bn1 = nn.BatchNorm1d(300)

        # 第二个隐藏层 / Second hidden layer
        self.fc2 = nn.Linear(300, 300)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='elu')
        self.elu2 = nn.ELU()
        self.bn2 = nn.BatchNorm1d(300)

        # 输出层 / Output layer
        self.fc3 = nn.Linear(300, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.bn_input(x)

        x = self.elu1(self.fc1(x))
        x = self.bn1(x)

        x = self.elu2(self.fc2(x))
        x = self.bn2(x)

        x = self.fc3(x)  # CrossEntropyLoss 包含 softmax
        return x

model_bn_after = ModelBNAfter().to(device)
print(model_bn_after)
print(f"\n总参数量: {sum(p.numel() for p in model_bn_after.parameters()):,}")

In [ ]:
# 查看 BatchNorm1d 层的参数 / Inspect BatchNorm1d layer parameters
bn_layer = model_bn_after.bn1

print("BatchNorm1d 层参数:")
print("=" * 50)

for name, param in bn_layer.named_parameters():
    trainable_status = "可训练" if param.requires_grad else "不可训练"
    print(f"{name:20s} | shape={list(param.shape)} | {trainable_status}")

print("\n缓冲区 (不可训练的统计量):")
for name, buf in bn_layer.named_buffers():
    print(f"{name:20s} | shape={list(buf.shape)} | 不可训练")

print("\n参数说明:")
print("- weight (γ): 缩放因子（可训练）")
print("- bias (β): 偏移量（可训练）")
print("- running_mean: 滑动均值（推理时使用，不可训练）")
print("- running_var: 滑动方差（推理时使用，不可训练）")
print("- num_batches_tracked: 已跟踪的 batch 数量")

## 2. 优化方式：激活函数前使用 BN

论文原作者建议在激活函数**之前**应用 BN，这样可以：
- 归一化线性组合的输出
- 避免 Linear 层的 bias 与 BN 的 beta 参数冗余

In [ ]:
class ModelBNBefore(nn.Module):
    """
    优化版本：BN 在激活函数之前
    Optimized model: BN before activation function.

    Architecture: Linear(bias=False) -> BatchNorm1d -> ELU -> ... -> Softmax

    使用 bias=False 因为 BN 的 beta 会替代 bias 的作用。
    Using bias=False because BN's beta replaces the role of bias.
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        # 输入归一化 / Input normalization
        self.bn_input = nn.BatchNorm1d(28 * 28)

        # Dense 层不使用 bias（BN 的 beta 会替代 bias 的作用）
        # Dense layer without bias (BN's beta replaces bias)
        self.fc1 = nn.Linear(28 * 28, 300, bias=False)
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='elu')
        self.bn1 = nn.BatchNorm1d(300)
        self.elu1 = nn.ELU()

        self.fc2 = nn.Linear(300, 300, bias=False)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='elu')
        self.bn2 = nn.BatchNorm1d(300)
        self.elu2 = nn.ELU()

        self.fc3 = nn.Linear(300, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.bn_input(x)

        x = self.elu1(self.bn1(self.fc1(x)))
        x = self.elu2(self.bn2(self.fc2(x)))

        x = self.fc3(x)
        return x

model_bn_before = ModelBNBefore().to(device)
print(model_bn_before)

# 对比参数量 / Compare parameter counts
params_after = sum(p.numel() for p in model_bn_after.parameters())
params_before = sum(p.numel() for p in model_bn_before.parameters())
print("\n参数量对比:")
print(f"BN 在激活后: {params_after:,} 参数")
print(f"BN 在激活前 (bias=False): {params_before:,} 参数")
print(f"节省参数量: {params_after - params_before:,} (来自省略的 Dense bias)")

## 3. 训练与推理的区别

BN 在训练和推理阶段有不同的行为：

| 阶段 | 均值/方差来源 | 说明 |
|------|---------------|------|
| 训练 | 当前 mini-batch | 实时计算，同时更新滑动统计量 |
| 推理 | 滑动均值/方差 (running_mean/running_var) | 使用训练期间累积的统计量 |

在 PyTorch 中，通过 `model.train()` 和 `model.eval()` 自动切换，无需手动传参。

In [ ]:
def demonstrate_bn_behavior():
    """
    演示 BatchNorm1d 在训练和推理阶段的不同行为
    Demonstrate different behavior of BatchNorm1d in training vs inference.
    """
    # 创建简单的 BN 层 / Create a simple BN layer
    bn = nn.BatchNorm1d(10)

    # 模拟输入数据 / Simulate input data
    x_train = torch.randn(32, 10)  # batch_size=32, features=10
    x_test = torch.randn(8, 10)    # 测试数据 / test data

    # 训练模式 / Training mode
    bn.train()
    output_train = bn(x_train)
    print("训练模式 (model.train()):")
    print(f"  输入均值: {x_train.mean().item():.4f}")
    print(f"  输出均值: {output_train.mean().item():.4f}")
    print(f"  输入标准差: {x_train.std().item():.4f}")
    print(f"  输出标准差: {output_train.std().item():.4f}")

    # 推理模式 / Inference mode
    bn.eval()
    output_inference = bn(x_test)
    print("\n推理模式 (model.eval()):")
    print(f"  使用 running_mean: {bn.running_mean[:3].tolist()}...")
    print(f"  使用 running_var:  {bn.running_var[:3].tolist()}...")
    print(f"  num_batches_tracked: {bn.num_batches_tracked.item()}")

    # 重要提醒 / Important reminder
    print("\n⚠️ 注意: PyTorch 中必须显式调用 model.eval() 切换到推理模式！")
    print("   否则 BN 层会继续使用 batch 统计量而非 running 统计量。")

demonstrate_bn_behavior()

## 4. 完整训练示例

使用 Fashion-MNIST 数据集，对比带 BN 和不带 BN 的模型训练效果。

In [ ]:
# 加载 Fashion MNIST 数据集 / Load Fashion-MNIST dataset via torchvision
transform = transforms.Compose([
    transforms.ToTensor(),  # 自动归一化到 [0, 1] / Automatically normalize to [0, 1]
])

# 下载并加载数据 / Download and load data
train_val_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# 划分训练集和验证集 / Split into training and validation sets
train_size = 55000
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_val_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"训练集: {len(train_dataset)} 样本")
print(f"验证集: {len(val_dataset)} 样本")
print(f"测试集: {len(test_dataset)} 样本")
print(f"图像形状: {train_dataset[0][0].shape}")

In [ ]:
class ModelWithBN(nn.Module):
    """
    带批量归一化的分类模型
    Classification model with Batch Normalization.

    BN 放在激活函数之前，Linear 层使用 bias=False。
    BN is placed before activation, Linear layers use bias=False.
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.bn_input = nn.BatchNorm1d(28 * 28)

        self.fc1 = nn.Linear(28 * 28, 256, bias=False)
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='elu')
        self.bn1 = nn.BatchNorm1d(256)
        self.elu1 = nn.ELU()

        self.fc2 = nn.Linear(256, 128, bias=False)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='elu')
        self.bn2 = nn.BatchNorm1d(128)
        self.elu2 = nn.ELU()

        self.fc3 = nn.Linear(128, 64, bias=False)
        nn.init.kaiming_normal_(self.fc3.weight, nonlinearity='elu')
        self.bn3 = nn.BatchNorm1d(64)
        self.elu3 = nn.ELU()

        self.fc4 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.bn_input(x)

        x = self.elu1(self.bn1(self.fc1(x)))
        x = self.elu2(self.bn2(self.fc2(x)))
        x = self.elu3(self.bn3(self.fc3(x)))

        x = self.fc4(x)
        return x


class ModelWithoutBN(nn.Module):
    """
    不带批量归一化的分类模型（对照组）
    Classification model without Batch Normalization (control group).
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(28 * 28, 256)
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='elu')
        self.elu1 = nn.ELU()

        self.fc2 = nn.Linear(256, 128)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='elu')
        self.elu2 = nn.ELU()

        self.fc3 = nn.Linear(128, 64)
        nn.init.kaiming_normal_(self.fc3.weight, nonlinearity='elu')
        self.elu3 = nn.ELU()

        self.fc4 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)

        x = self.elu1(self.fc1(x))
        x = self.elu2(self.fc2(x))
        x = self.elu3(self.fc3(x))

        x = self.fc4(x)
        return x


# 创建两个模型进行对比 / Create two models for comparison
model_with_bn = ModelWithBN().to(device)
model_without_bn = ModelWithoutBN().to(device)

params_bn = sum(p.numel() for p in model_with_bn.parameters())
params_no_bn = sum(p.numel() for p in model_without_bn.parameters())
print(f"带 BN 的模型参数量: {params_bn:,}")
print(f"不带 BN 的模型参数量: {params_no_bn:,}")
print(f"BN 额外参数量: {params_bn - params_no_bn:,} (来自 BN 的 weight, bias, 以及省略的 Linear bias)")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001):
    """
    训练模型并记录训练历史
    Train model and record training history.

    Parameters:
    -----------
    model : nn.Module
        要训练的模型 / Model to train
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    val_loader : DataLoader
        验证数据加载器 / Validation data loader
    epochs : int
        训练轮数 / Number of training epochs
    lr : float
        学习率 / Learning rate

    Returns:
    --------
    dict : 包含训练和验证的损失与准确率历史
        Dictionary containing training and validation loss/accuracy history
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    for epoch in range(epochs):
        # === 训练阶段 / Training phase ===
        model.train()  # 重要：启用 BN 的训练模式 / Important: enable BN training mode
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / total
        train_acc = correct / total

        # === 验证阶段 / Validation phase ===
        model.eval()  # 重要：切换 BN 到推理模式 / Important: switch BN to inference mode
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    return history

In [ ]:
# 创建数据加载器 / Create data loaders
EPOCHS = 10
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 训练带 BN 的模型 / Train model with BN
print("=" * 60)
print("训练带 BN 的模型...")
print("=" * 60)
history_with_bn = train_model(
    model_with_bn, train_loader, val_loader, epochs=EPOCHS, lr=0.001
)

# 训练不带 BN 的模型 / Train model without BN
print("\n" + "=" * 60)
print("训练不带 BN 的模型...")
print("=" * 60)
history_without_bn = train_model(
    model_without_bn, train_loader, val_loader, epochs=EPOCHS, lr=0.001
)

In [ ]:
# 绘制训练曲线对比 / Plot training curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 准确率对比 / Accuracy comparison
axes[0].plot(history_with_bn['train_acc'], 'b-', label='带 BN (训练)')
axes[0].plot(history_with_bn['val_acc'], 'b--', label='带 BN (验证)')
axes[0].plot(history_without_bn['train_acc'], 'r-', label='不带 BN (训练)')
axes[0].plot(history_without_bn['val_acc'], 'r--', label='不带 BN (验证)')
axes[0].set_title('准确率对比', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 损失对比 / Loss comparison
axes[1].plot(history_with_bn['train_loss'], 'b-', label='带 BN (训练)')
axes[1].plot(history_with_bn['val_loss'], 'b--', label='带 BN (验证)')
axes[1].plot(history_without_bn['train_loss'], 'r-', label='不带 BN (训练)')
axes[1].plot(history_without_bn['val_loss'], 'r--', label='不带 BN (验证)')
axes[1].set_title('损失对比', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('batch_normalization_comparison_pytorch.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 测试集评估 / Evaluate on test set
def evaluate_model(model, test_loader):
    """
    在测试集上评估模型
    Evaluate model on test set.
    """
    model.eval()  # 切换到推理模式 / Switch to inference mode
    criterion = nn.CrossEntropyLoss()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return test_loss / total, correct / total

test_loss_bn, test_acc_bn = evaluate_model(model_with_bn, test_loader)
test_loss_no_bn, test_acc_no_bn = evaluate_model(model_without_bn, test_loader)

print("测试集评估:")
print(f"带 BN:   准确率 = {test_acc_bn:.4f}, 损失 = {test_loss_bn:.4f}")
print(f"不带 BN: 准确率 = {test_acc_no_bn:.4f}, 损失 = {test_loss_no_bn:.4f}")
print(f"\nBN 带来的准确率提升: {(test_acc_bn - test_acc_no_bn) * 100:.2f}%")

## 5. BN 的超参数配置

In [ ]:
# BatchNorm1d 的完整参数 / Full parameters of BatchNorm1d
bn_layer_custom = nn.BatchNorm1d(
    num_features=300,       # 归一化的特征数 / Number of features to normalize
    eps=1e-3,               # 防止除零的小常数（默认 1e-5）/ Small constant for numerical stability (default 1e-5)
    momentum=0.1,           # 滑动平均动量（默认 0.1）/ Momentum for running stats (default 0.1)
    affine=True,            # 是否使用可学习的 γ 和 β / Whether to use learnable γ and β
    track_running_stats=True # 是否跟踪 running_mean 和 running_var / Whether to track running stats
)

print("BatchNorm1d 超参数说明:")
print("=" * 60)
print("num_features: 归一化的特征维度（必须指定）")
print("  - 全连接层: 等于 Linear 的输出维度")
print("  - 卷积层: 等于 Conv 的输出通道数")
print()
print("momentum: 滑动平均动量")
print("  - PyTorch 默认 0.1（等价于 Keras 的 0.9）")
print("  - 更新公式: running = (1 - momentum) * running + momentum * batch")
print("  - 0.1: 适合一般数据集（默认）")
print("  - 0.01: 适合大数据集（等价于 Keras 的 0.99）")
print()
print("eps: 数值稳定性常数，通常无需修改")
print("  - PyTorch 默认 1e-5（Keras 默认 1e-3）")
print()
print("affine: 是否使用可学习的 γ (weight) 和 β (bias)")
print("  - True (默认): 使用可学习的缩放和偏移")
print("  - False: 不使用，归一化后直接输出")
print()
print("track_running_stats: 是否跟踪 running 统计量")
print("  - True (默认): 训练时累积 running_mean/running_var")
print("  - False: 推理时也使用 batch 统计量")

In [ ]:
# 演示 momentum 参数的影响 / Demonstrate the effect of momentum parameter
def demonstrate_momentum():
    """
    演示不同 momentum 值对 running_mean 更新的影响
    Demonstrate the effect of different momentum values on running_mean updates.

    PyTorch 更新公式: running = (1 - momentum) * running + momentum * batch_mean
    Keras 更新公式:   moving  = momentum * moving + (1 - momentum) * batch_mean
    """
    torch.manual_seed(42)

    # 创建不同 momentum 的 BN 层 / Create BN layers with different momentum
    bn_default = nn.BatchNorm1d(5, momentum=0.1)   # PyTorch 默认
    bn_small = nn.BatchNorm1d(5, momentum=0.01)    # 等价 Keras 默认 0.99
    bn_large = nn.BatchNorm1d(5, momentum=0.3)     # 更快适应

    layers = {'momentum=0.1 (默认)': bn_default,
              'momentum=0.01 (≈Keras 0.99)': bn_small,
              'momentum=0.3': bn_large}

    # 模拟训练过程 / Simulate training process
    print("模拟 5 个 batch 的训练，观察 running_mean 变化:")
    print("=" * 70)

    for i in range(5):
        x = torch.randn(32, 5) * (i + 1)  # 逐渐增大的数据 / Gradually increasing data
        batch_mean = x.mean(dim=0)

        print(f"\nBatch {i+1} - 真实均值: {batch_mean[:2].tolist()}")

        for name, layer in layers.items():
            layer.train()
            _ = layer(x)
            print(f"  {name:30s} running_mean: {layer.running_mean[:2].tolist()}")

    print("\n结论: momentum 越小，running_mean 变化越慢（保留更多历史信息）")

demonstrate_momentum()

## 6. 使用建议

### 何时使用 BN

| 场景 | 建议 |
|------|------|
| 深层网络 | 强烈推荐 |
| 训练不稳定 | 推荐 |
| 想用更大学习率 | 推荐 |
| 小 batch size (<16) | 考虑使用 Layer Normalization |
| RNN/LSTM | 使用 Layer Normalization |

### 注意事项

1. **Batch Size 影响**: 小 batch 时 BN 效果不稳定，考虑使用 Layer Normalization
2. **Dropout + BN**: 通常不建议同时使用，或将 Dropout 放在 BN 之后
3. **迁移学习**: 微调时可能需要冻结 BN 层的统计量（设置 `bn.eval()` 或 `bn.track_running_stats=False`）
4. **推理模式**: PyTorch 中必须显式调用 `model.eval()`，否则 BN 会使用 batch 统计量

In [ ]:
# 验证代码正确性 / Verify code correctness
print("批量归一化模块测试完成 (PyTorch 版本)")
print("\n关键要点:")
print("1. BN 归一化每层输入，加速训练并稳定梯度")
print("2. 推荐在激活函数前使用 BN，并设置 bias=False")
print("3. PyTorch 中通过 model.train()/eval() 切换训练/推理模式")
print("4. PyTorch momentum 与 Keras momentum 互为补数！0.1 ≈ Keras 0.9")
print("5. 小 batch size 时考虑使用 Layer Normalization")

## 练习

### 练习 1：卷积网络中的 BN

将本 notebook 中的全连接网络改为卷积网络（使用 `nn.Conv2d` + `nn.BatchNorm2d`），在 Fashion-MNIST 上训练并对比有/无 BN 的效果。

提示：
- `nn.BatchNorm2d(num_features)` 中 `num_features` 等于 `Conv2d` 的 `out_channels`
- 输入不需要 Flatten，保持 `[batch, channel, height, width]` 形状
- 卷积层同样建议使用 `bias=False` 配合 BN

### 练习 2：不同 momentum 值的影响

分别使用 `momentum=0.01`、`momentum=0.1`（默认）、`momentum=0.3` 训练同一个模型，对比训练曲线和最终测试准确率。思考：为什么大数据集通常使用更小的 momentum？

### 练习 3：忘记 model.eval() 的后果

在评估测试集时，故意不调用 `model.eval()`（即保持 `model.train()` 模式），观察测试准确率的变化。解释为什么推理时必须使用 `model.eval()`，特别是当 batch size 很小时。